<a href="https://colab.research.google.com/github/marcopannullo1-source/Finance/blob/main/S%26P500_new_entry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install yfinance pandas numpy requests

In [47]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from typing import List, Dict, Optional
import requests
import io

# ---------------------------
# PARAMETRI DI SOGLIA
# ----------------------------
MIN_UNADJ_MCAP_USD = 22.7e9          # 22,7 miliardi USD
MIN_FLOAT_MCAP_USD = MIN_UNADJ_MCAP_USD * 0.50  # 50% → 11,35 mld
MIN_FLOAT_PCT = 0.10                 # flottante ≥ 10%
MIN_MONTHLY_VOLUME = 250_000         # azioni/mese (ultimi 6 mesi)
MIN_DV_FLOAT_RATIO = 0.75            # DV/float-cap ≥ 0.75
MIN_SEASONING_MONTHS = 12            # 12 mesi di listing

# ---------------------------
# HELPERS PER SCRAPING WIKIPEDIA
# ----------------------------
def _get_tickers_from_wikipedia_table(url: str, expected_column_name: str) -> List[str]:
    """
    General function to scrape tickers from a specific table on a Wikipedia page,
    searching for the table containing the expected column name.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        print(f"Attempting to download tickers from Wikipedia: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        tables = pd.read_html(io.StringIO(response.text))

        for i, df_candidate in enumerate(tables):
            # Flatten multi-level columns if they exist
            if isinstance(df_candidate.columns, pd.MultiIndex):
                df_candidate.columns = [' '.join(col).strip() for col in df_candidate.columns.values]

            # Clean column names to handle potential whitespace or special characters
            df_candidate.columns = df_candidate.columns.astype(str).str.strip()

            if expected_column_name in df_candidate.columns:
                tickers = df_candidate[expected_column_name].tolist()
                # Normalize ticker names for yfinance (e.g., BRK.B -> BRK-B)
                tickers = [t.replace('.', '-') for t in tickers]
                print(f"Successfully downloaded {len(tickers)} tickers from {url} (table index {i}).")
                return tickers

        # If loop finishes, column not found in any table
        raise ValueError(f"Column '{expected_column_name}' not found in any table on {url}.")

    except Exception as e:
        print(f"Error scraping tickers from {url}: {e}")
        return []

# ---------------------------
# FUNZIONE PRINCIPALE
# ----------------------------
def get_sp500_eligibility(ticker: str, reference_date: Optional[datetime] = None) -> Dict:
    """
    Restituisce un dizionario con i criteri di ammissibilità S&P 500 per un ticker.
    """
    if reference_date is None:
        reference_date = datetime.now()

    # Normalize ticker for yfinance if necessary
    yf_ticker = ticker.replace('.', '-')

    tick = yf.Ticker(yf_ticker)
    info = tick.info  # dizionario con metadati

    # 1. Sede e listing
    country = info.get('country', '')
    exchange = info.get('exchange', '')
    is_us = country == 'United States'
    is_nyse_nasdaq = exchange in ['NYQ', 'NMS']  # NYSE, NASDAQ (codici yfinance)

    # 2. Capitalizzazione (unadjusted) e flottante
    market_cap = info.get('marketCap', None)  # in USD
    shares_outstanding = info.get('sharesOutstanding', None)
    float_shares = info.get('floatShares', None)  # a volte assente

    # Calcolo flottante % se disponibile
    if shares_outstanding and float_shares:
        float_pct = float_shares / shares_outstanding
    else:
        float_pct = None

    float_mcap = None
    if market_cap and float_pct is not None:
        float_mcap = market_cap * float_pct

    # 3. Liquidità volumi mensili (ultimi 6 mesi)
    hist = tick.history(period='1y')
    if len(hist) < 180:  # ~6 mesi
        monthly_volume_ok = None
        dv_float_ratio = None
    else:
        # Raggruppa per mese
        hist_m = hist['Volume'].resample('ME').sum()
        last_6_months = hist_m.tail(6)
        monthly_volume_ok = (last_6_months >= MIN_MONTHLY_VOLUME).all()

        # Dollar value traded (ultimi 12 mesi) / float-cap
        hist_12m = tick.history(period='12mo')
        dollar_value = (hist_12m['Volume'] * hist_12m['Close']).sum()
        if float_mcap and float_mcap > 0:
            dv_float_ratio = dollar_value / float_mcap
        else:
            dv_float_ratio = None

    # 4. Redditività GAAP (ultimi 4 trimestri)
    try:
        quarterly = tick.quarterly_financials
        if 'Net Income' in quarterly.index:
            ni_q = quarterly.loc['Net Income'].dropna()
            last_4q = ni_q.head(4)
            gaap_profitable = (last_4q > 0).all() and (last_4q.iloc[-1] > 0)
        else:
            gaap_profitable = None
    except Exception:
        gaap_profitable = None

    # 5. Periodo di quotazione (IPO date approssimato)
    if len(hist) > 0:
        first_date = hist.index[0].to_pydatetime()
        seasoning_months = (reference_date.year - first_date.year)*12 + (reference_date.month - first_date.month)
        seasoning_ok = seasoning_months >= MIN_SEASONING_MONTHS
    else:
        seasoning_months = None
        seasoning_ok = None

    # 6. Momentum
    mom_1w = None # Initialize mom_1w
    if not hist.empty and 'Close' in hist.columns:
        prices = hist['Close']
        if len(prices) >= 7:
            mom_1w_pct  = prices.iloc[-1] / prices.iloc[-7]  - 1
            mom_1w = mom_1w_pct * 100

    # Costruisci output
    result = {
        'ticker': ticker,
        'country': country,
        'exchange': exchange,
        'is_us': is_us,
        'is_nyse_nasdaq': is_nyse_nasdaq,
        'market_cap_usd': market_cap,
        'momentum_1w': mom_1w,
        'float_pct': float_pct,
        'float_mcap_usd': float_mcap,
        'min_unadj_mcap_ok': market_cap is not None and market_cap >= MIN_UNADJ_MCAP_USD,
        'min_float_mcap_ok': float_mcap is not None and float_mcap >= MIN_FLOAT_MCAP_USD,
        'min_float_pct_ok': float_pct is not None and float_pct >= MIN_FLOAT_PCT,
        'monthly_volume_last_6m_ok': monthly_volume_ok,
        'dv_float_ratio': dv_float_ratio,
        'min_dv_float_ratio_ok': dv_float_ratio is not None and dv_float_ratio >= MIN_DV_FLOAT_RATIO,
        'gaap_profitable': gaap_profitable,
        'seasoning_months': seasoning_months,
        'seasoning_ok': seasoning_ok,
    }

    # Flag finale (tutti i criteri numerici soddisfatti)
    numeric_checks = [
        result['is_us'],
        result['is_nyse_nasdaq'],
        result['min_unadj_mcap_ok'],
        result['min_float_mcap_ok'],
        result['min_float_pct_ok'],
        result['monthly_volume_last_6m_ok'],
        result['min_dv_float_ratio_ok'],
        result['gaap_profitable'],
        result['seasoning_ok'],
    ]
    result['eligible_sp500'] = all(bool(c) for c in numeric_checks)

    return result


# ---------------------------
# SCARICAMENTO LISTA TICKER S&P 500 (componenti attuali)
# ----------------------------
def get_sp500_components_from_wikipedia() -> List[str]:
    """
    Tenta di scaricare i ticker dei componenti S&P 500 dalla pagina Wikipedia.
    In caso di errore, tornerà a una lista hardcoded di noti componenti S&P 500.
    """
    sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    try:
        # Reuse the generic scraping function
        sp500_tickers = _get_tickers_from_wikipedia_table(sp500_url, expected_column_name='Symbol')
        if sp500_tickers:
            return sp500_tickers
        else:
            raise Exception("Failed to get S&P 500 tickers from Wikipedia.")
    except Exception as e:
        print(f"Error scraping S&P 500 components from Wikipedia: {e}.")
        print("Falling back to a small hardcoded list of known S&P 500 components (for demonstration).")
        return ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA', 'JPM', 'JNJ', 'XOM', 'TSLA', 'UNH']


# ---------------------------
# SCARICAMENTO LISTA TICKER (universo di candidati)
# ----------------------------
def get_broad_us_ticker_universe() -> List[str]:
    """
    Dynamically fetches a broad list of US large-cap tickers from Wikipedia pages
    for major US indices (S&P 500, NASDAQ-100, S&P 100, S&P MidCap 400) to serve as a candidate universe.
    """
    print("Dynamically fetching a broad US ticker universe from Wikipedia...")
    all_tickers = []

    # Get S&P 500 components
    sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    sp500_tickers = _get_tickers_from_wikipedia_table(sp500_url, expected_column_name='Symbol')
    all_tickers.extend(sp500_tickers)

    # Get NASDAQ-100 components
    nasdaq100_url = 'https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies'
    nasdaq100_tickers = _get_tickers_from_wikipedia_table(nasdaq100_url, expected_column_name='Ticker')
    all_tickers.extend(nasdaq100_tickers)

    # Get S&P 100 components
    sp100_url = 'https://en.wikipedia.org/wiki/S%26P_100'
    sp100_tickers = _get_tickers_from_wikipedia_table(sp100_url, expected_column_name='Symbol')
    all_tickers.extend(sp100_tickers)

    # Get S&P MidCap 400 components
    sp400_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_400_companies'
    sp400_tickers = _get_tickers_from_wikipedia_table(sp400_url, expected_column_name='Symbol')
    all_tickers.extend(sp400_tickers)


    unique_tickers = list(set(all_tickers))
    if not unique_tickers:
        print("Failed to fetch any tickers from Wikipedia. Falling back to a small hardcoded list.")
        # Fallback to a small known S&P 500 list if scraping fails for all sources
        unique_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'JPM', 'V', 'JNJ', 'UNH', 'PG']

    print(f"Broad US ticker universe constructed with {len(unique_tickers)} unique tickers.")
    return unique_tickers


# ---------------------------
# SCREENING AUTONOMO
# ----------------------------
def auto_screen_sp500_eligible(top_n: int = 2000, reference_date: Optional[datetime] = None,
                                max_tickers_to_scan: Optional[int] = None) -> pd.DataFrame:
    """
    Scarica autonomamente una lista di ticker US che NON sono nell'S&P 500,
    esegue la checklist S&P 500 e restituisce i primi 'top_n' eleggibili
    (ordinati per market cap decrescente) come potenziali nuovi ingressi.

    Parametri:
    - top_n: numero di ticker da restituire (default 2000)
    - max_tickers_to_scan: limite opzionale al numero di ticker da scansionare (per test)
    """
    print("Preparazione della lista di ticker per lo screening...")

    # 1. Ottieni l'universo di ticker da cui cercare potenziali candidati
    universe_tickers = get_broad_us_ticker_universe()
    print(f"Totale ticker nell'universo di screening: {len(universe_tickers)}")

    # 2. Ottieni i componenti attuali dell'S&P 500
    current_sp500_components = get_sp500_components_from_wikipedia()

    # 3. Filtra i ticker che non sono nell'S&P 500
    potential_new_candidates = list(set(universe_tickers) - set(current_sp500_components))
    print(f"Totale potenziali nuovi candidati S&P 500 (non attuali membri): {len(potential_new_candidates)}")

    all_tickers_to_screen = potential_new_candidates

    if max_tickers_to_scan:
        all_tickers_to_screen = all_tickers_to_screen[:max_tickers_to_scan]
        print(f"Limitato a {max_tickers_to_scan} ticker per scanning.")

    results = []
    for i, t in enumerate(all_tickers_to_screen):
        try:
            res = get_sp500_eligibility(t, reference_date)
            results.append(res)
            # Debug print: Show all eligibility results for each ticker
            print(f"Ticker: {t}, Results: {res}")

            # Progresso
            if (i + 1) % 100 == 0:
                print(f"Scansionati {i+1}/{len(all_tickers_to_screen)} ticker...")
        except Exception as e:
            print(f"Errore su {t}: {e}") # Keep this print for individual ticker errors
            pass

    df = pd.DataFrame(results)
    if df.empty:
        print("Nessun dato elaborato.")
        return pd.DataFrame()

    cols = [
        'ticker', 'is_us', 'is_nyse_nasdaq', 'market_cap_usd', 'momentum_1w', 'float_pct',
        'min_unadj_mcap_ok', 'min_float_mcap_ok', 'min_float_pct_ok',
        'monthly_volume_last_6m_ok', 'dv_float_ratio', 'min_dv_float_ratio_ok',
        'gaap_profitable', 'seasoning_months', 'seasoning_ok', 'eligible_sp500'
    ]
    df = df[cols]

    # Filtra solo eleggibili
    eligible_df = df[df['eligible_sp500'] == True].copy()

    if eligible_df.empty:
        print("Nessun ticker soddisfa tutti i criteri.")
        return pd.DataFrame()

    # Ordina per market cap decrescente
    eligible_df = eligible_df.sort_values('market_cap_usd', ascending=False)

    eligible_df = eligible_df.sort_values('momentum_1w', ascending=False)

    # Restituisci i primi top_n
    result = eligible_df.head(top_n).reset_index(drop=True)
    print(f"\nTrovati {len(eligible_df)} ticker eleggibili. Mostrati i primi {len(result)}.")

    return result


# ---------------------------
# ESEMPIO DI UTILIZZO
# ----------------------------
if __name__ == '__main__':
    print("=== S&P 500 Eligibility Auto-Screening ===\n")

    # Esegui screening autonomo
    # Per full scan va rimosso max_tickers_to_scan, per test veloce va inserito;
    potential_sp500_candidates = auto_screen_sp500_eligible(top_n=2000) #, max_tickers_to_scan=500)

    if len(potential_sp500_candidates) > 0:
        print(f"\n=== PRIMI {len(potential_sp500_candidates)} POTENZIALI CANDIDATI S&P 500 ===\n")
        display(potential_sp500_candidates)
        potential_sp500_candidates.to_excel("potential_sp500_candidates.xlsx", index=False)

=== S&P 500 Eligibility Auto-Screening ===

Preparazione della lista di ticker per lo screening...
Dynamically fetching a broad US ticker universe from Wikipedia...
Attempting to download tickers from Wikipedia: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
Successfully downloaded 503 tickers from https://en.wikipedia.org/wiki/List_of_S%26P_500_companies (table index 0).
Attempting to download tickers from Wikipedia: https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies
Successfully downloaded 102 tickers from https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies (table index 0).
Attempting to download tickers from Wikipedia: https://en.wikipedia.org/wiki/S%26P_100
Successfully downloaded 101 tickers from https://en.wikipedia.org/wiki/S%26P_100 (table index 2).
Attempting to download tickers from Wikipedia: https://en.wikipedia.org/wiki/List_of_S%26P_400_companies
Successfully downloaded 400 tickers from https://en.wikipedia.org/wiki/List_of_S%26P_400_companies (t

,ticker,is_us,is_nyse_nasdaq,market_cap_usd,momentum_1w,float_pct,min_unadj_mcap_ok,min_float_mcap_ok,min_float_pct_ok,monthly_volume_last_6m_ok,dv_float_ratio,min_dv_float_ratio_ok,gaap_profitable,seasoning_months,seasoning_ok,eligible_sp500
0,OKTA,True,True,2.776768e+10,21.610142,0.997163,True,True,True,True,2.948528,True,True,12,True,True
1,ALNY,True,True,3.530353e+10,11.789754,0.995720,True,True,True,True,3.350547,True,True,12,True,True
2,ROKU,True,True,2.343646e+10,-0.233908,0.991620,True,True,True,True,4.222714,True,True,12,True,True
3,USFD,True,True,2.296891e+10,-2.921276,0.992830,True,True,True,True,2.356943,True,True,12,True,True
4,CRS,True,True,2.292629e+10,-2.996437,0.976480,True,True,True,True,3.588840,True,True,12,True,True
5,ATI,True,True,2.738863e+10,-4.134695,0.990160,True,True,True,True,2.369606,True,True,12,True,True
6,ALAB,True,True,4.685314e+10,-4.450697,0.898000,True,True,True,True,7.282949,True,True,12,True,True
7,SN,True,True,2.471158e+10,-4.576753,0.609871,True,True,True,True,3.654656,True,True,12,True,True
8,ILMN,True,True,3.172812e+10,-5.868652,0.990280,True,True,True,True,2.086516,True,True,12,True,True
9,P,True,True,3.070092e+10,-8.044605,0.947506,True,True,True,True,2.302411,True,True,12,True,True
